# Final model comparison

This notebook consolidates final temporal-test performance. It does not retrain models or select features.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import yaml
import joblib
import pandas as pd
from IPython.display import display

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.final_comparison import (
    evaluate_models_on_table, add_feature_metadata, rank_models, save_final_comparison,
)

In [3]:
prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.parquet'
if not prediction_path.exists():
    prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.csv'
predictions = load_admission_file(prediction_path)
target = config['columns']['target']
target_mapping = config.get('target_mapping', config.get('analysis', {}).get('target_mapping', {}))
test = predictions[predictions['split'].eq('test')].copy()
prediction_columns = [c for c in test.columns if c.endswith('_prob')]
print(f'Test admissions: {len(test):,}')
print(f'Model prediction columns: {len(prediction_columns)}')

Test admissions: 2,483
Model prediction columns: 6


In [4]:
comparison = evaluate_models_on_table(
    test, target_column=target, prediction_columns=prediction_columns,
    target_mapping=target_mapping,
    threshold=config.get('analysis', {}).get('evaluation_threshold', 0.5),
)
model_configs = config.get('models', [])
feature_counts = {}
for model_cfg in model_configs:
    path = Path(config['outputs']['model_directory']) / f"{model_cfg['name']}.joblib"
    if path.exists():
        bundle = joblib.load(path)
        if isinstance(bundle, dict) and bundle.get('features') is not None:
            feature_counts[model_cfg['name']] = len(bundle['features'])
comparison = add_feature_metadata(comparison, model_configs=model_configs, feature_counts=feature_counts)
ranked = rank_models(comparison)
display(ranked)

,model,prediction_column,n,n_events,event_rate,auroc,auprc,brier_score,threshold,tp,...,balanced_accuracy,f1,calibration_intercept,calibration_slope,observed_expected_ratio,feature_set,model_type,n_features,primary_rank,composite_score
0,xgb_kim_score,xgb_kim_score_prob,2483,508,0.204591,0.495556,0.203157,0.162869,0.5,0,...,0.500000,0.000000,-0.441228,0.636857,1.067865,kim_score,xgboost,5,4.0,2.333333
1,lr_all_scores,lr_all_scores_prob,2483,508,0.204591,0.488996,0.198725,0.248381,0.5,241,...,0.489103,0.278774,-1.353105,0.372360,0.411947,all_scores,logistic_regression,6,6.0,2.333333
2,xgb_all_scores,xgb_all_scores_prob,2483,508,0.204591,0.509264,0.210962,0.163140,0.5,0,...,0.500000,0.000000,-1.065966,0.201427,1.071928,all_scores,xgboost,6,3.0,1.666667
3,lr_kim_score,lr_kim_score_prob,2483,508,0.204591,0.495556,0.203157,0.249759,0.5,447,...,0.495404,0.329768,-1.357386,1.077531,0.409520,kim_score,logistic_regression,5,4.0,1.500000
4,xgb_clinical_baseline,xgb_clinical_baseline_prob,2483,508,0.204591,0.514920,0.219864,0.164069,0.5,0,...,0.500000,0.000000,-1.033741,0.220184,1.076944,clinical_baseline,xgboost,16,2.0,1.000000
5,lr_clinical_baseline,lr_clinical_baseline_prob,2483,508,0.204591,0.521869,0.215387,0.248659,0.5,259,...,0.508339,0.297530,-1.353805,0.544682,0.411742,clinical_baseline,logistic_regression,16,1.0,0.666667


In [5]:
output_dir = Path(config['outputs'].get('final_directory', 'results/final'))
output_dir.mkdir(parents=True, exist_ok=True)
ranked.to_csv(output_dir / 'final_model_comparison.csv', index=False)
ranked.to_csv(output_dir / 'final_model_ranking.csv', index=False)
print(f'Saved results to {output_dir.resolve()}')

Saved results to /app/notebooks/results/final
